# 04 - Continuous Time Grid & Lag Structure Audit

This notebook consolidates all 24 monthly 15-minute demand files, audits time continuity across month boundaries, and inspects engineered lag features and target surge deltas ($\Delta y_t$).

In [ ]:
import sys
from pathlib import Path

# Resolve project root dynamically regardless of notebook working directory
PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f'Project root added to sys.path: {PROJECT_ROOT}')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from src.features import consolidate_demand_matrices, engineer_feature_lags
from src.config import PROCESSED_DATA_DIR

print('Imports successful.')

### 1. Load and Consolidate 24 Monthly Matrices
Merges individual monthly files from `data/processed/` and drops seam-boundary collisions.

In [ ]:
df_consolidated = consolidate_demand_matrices()
print(f'Total Consolidated Records: {len(df_consolidated):,}')

df_consolidated.groupby('venue')['timestamp_15m'].agg(['min', 'max', 'count'])

### 2. Time Grid Continuity Audit
Verifies that each step is strictly separated by exactly 15 minutes with 0 gaps or irregular hops.

In [ ]:
for venue_name, group in df_consolidated.groupby('venue'):
    time_diffs = group['timestamp_15m'].diff().dropna()
    irregular_steps = (time_diffs != pd.Timedelta('15min')).sum()
    print(f'Venue: {venue_name:<12} | Irregular intervals: {irregular_steps} (Expected: 0)')
    assert irregular_steps == 0, f'Found discontinuous intervals in {venue_name}!'

### 3. Feature Lag & Surge Delta Computation
Computes cyclical calendar features (`sin`/`cos`), momentum lags ($t-1, t-2, t-4, t-96, t-672$), rolling statistics, and target delta $\Delta y_t$.

In [ ]:
featured_df = engineer_feature_lags(df_consolidated)

# Drop the initial 7-day warmup period where lag_672 (1-week lag) is NaN
ml_ready = featured_df.dropna(subset=['demand_lag_672']).reset_index(drop=True)

display_cols = [
    'timestamp_15m', 'venue', 'weighted_demand',
    'demand_lag_1', 'demand_lag_4', 'demand_lag_672', 'target_delta_y'
]
print(f'ML-Ready matrix shape: {ml_ready.shape}')
ml_ready[display_cols].head(10)

### 4. Summary Statistics for Target Surge Delta ($\Delta y_t$)

In [ ]:
ml_ready.groupby('venue')['target_delta_y'].describe().round(2)